# Feature Engineering for TeacherGAIA Chatlog Dataset

Adapts the Google Search feature extraction pipeline to the TeacherGAIA dataset.

**Columns used:**
- `studId` — student identifier (same `studId` = same student)
- `prompt` — text entered by the student

All other columns (including `chatId`, `timestamp`, `response`) are **ignored** as instructed.

> Because there is no session/episode structure from `chatId`, features that were originally
> episode-level in the Google Search pipeline are adapted as follows:
> - **Lexical diversity, semantic diversity, entity diversity, topic diversity** → computed over **all prompts per student** (student-level).
> - **Hypernym, Brysbaert, Category nouns, Question/Closure/Comparative** → computed **per prompt**, then averaged per student.
> - **Levenshtein reformulation** → computed between **consecutive prompts per student** (ordered by row position).
> - **Time interval / Time total** → **not applicable** (no timestamp used); columns omitted.
> - **num_query_episode** → replaced by **total_prompts** (total prompts per student).

**Output files:**
- `TeacherGAIA_features_per_prompt.csv` — one row per prompt with all per-prompt features
- `TeacherGAIA_features_per_student.csv` — one row per student with student-average features

**Features extracted:**

| # | Feature | Dimension | Key output columns |
|---|---------|-----------|--------------------|
| 1 | Query Length | Specific-Broad | `query_length`, `avg_query_length` |
| 2 | Lexical Diversity | Specific-Broad | `lexical_diversity` (student-level Shannon entropy) |
| 3 | Semantic Diversity | Specific-Broad | `semantic_diversity_mean`, `semantic_diversity_var` (student-level) |
| 4 | Entity vs Category | Specific-Broad | `entity_category_diversity`, `specific_entity_diversity` (student-level) |
| 5 | Topic Diversity | Specific-Broad | `topic`, `topic_diversity`, `topic_count` (student-level) |
| 6 | Hypernym | Specific-Broad | `hypernym_distance`, `hypernym_specificity`, `student_hypernym_specificity_ratio` |
| 7 | Brysbaert Concreteness | Specific-Broad | `avg_text_concreteness`, `brysbaert_specificity`, `student_brysbaert_specificity_ratio` |
| 8 | Category Nouns | Specific-Broad | `Category_nouns`, `Category_noun_words` |
| 9 | Question Detection | Surface-Deep | `is_question`, `user_question_rate` |
| 10 | Closure Detection | Surface-Deep | `closure`, `closure_words`, `user_closure_rate` |
| 11 | Comparative Detection | Surface-Deep | `comparative`, `comparative_words`, `user_comparative_rate` |
| 12 | Reformulation (Levenshtein) | Surface-Deep | `lev_dist`, `lev_dist_norm` |
| 13 | Num Prompts Per Student | Surface-Deep | `total_prompts` |
| 14 | O_NP | Surface-Deep | `O_NP` |
| 15 | O_P | Surface-Deep | `O_P` |
| 16 | C_NP | Surface-Deep | `C_NP` |
| 17 | C_P | Surface-Deep | `C_P` |


## 1. Load & Preprocess Data

In [1]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

# ── CONFIG ───────────────────────────────────────────────────────────────────
INPUT_FILE       = 'TeacherGAIA_chatlog_filtered_Deidentified.xlsx'
OUTPUT_PROMPT    = 'TeacherGAIA_features_per_prompt.csv'
OUTPUT_USER      = 'TeacherGAIA_features_per_student.csv'
TOPIC_MODEL_PATH = 'google_query_2_model_100.pickle'   # from Serena Wan Jun Wei
# ─────────────────────────────────────────────────────────────────────────────

df_raw = pd.read_excel(INPUT_FILE)
print('Raw shape:', df_raw.shape)
print('Columns:', df_raw.columns.tolist())
df_raw.head(3)


Raw shape: (14829, 6)
Columns: ['id', 'prompt', 'response', 'timestamp', 'studId', 'chatId']


,id,prompt,response,timestamp,studId,chatId
0,20377,what should i eat today?\n,"As a teacher of Korean history and culture, I ...",2024-03-05 09:59:00,399,135
1,20384,recipe for kimchi\n,I can certainly guide you towards making your ...,2024-03-05 10:00:00,399,135
2,20456,basic greetings in korea\n,"Absolutely! In Korean culture, greetings are v...",2024-03-05 10:05:00,399,135


### (a) Drop Empty Prompts

In [2]:
# Create copy
df = df_raw.copy()

# Remove rows with empty prompts
df = df.dropna(subset=['prompt'])


### (b) Clean Whitespace

In [3]:
df['prompt'] = df['prompt'].astype(str).str.strip()
df['response'] = df['response'].astype(str).str.strip()

### (c) Remove Empty / Whitespace-Only Prompts

In [4]:
df = df[df['prompt'].str.len() > 0]

# remove prompts that are ONLY punctuation/emojis/symbols
df = df[~df['prompt'].str.fullmatch(r'[\W_]+', na=False)]

### (d) Remove Rows with Chinese Text

In [5]:
# Step 1: detect chinese text (character-based)
import re

CJK_PATTERN = re.compile(
    r'[\u3400-\u4DBF\u4E00-\u9FFF\uF900-\uFAFF]'
)

def has_chinese(text):
    if pd.isna(text):
        return False
    return bool(CJK_PATTERN.search(str(text)))

df['prompt_cn'] = df['prompt'].apply(has_chinese)
df['response_cn'] = df['response'].apply(has_chinese)


# Step 3: remove only rows where either prompt and response are chinese
df_clean = df[~(df['prompt_cn'] | df['response_cn'])].copy()

### (f) Reset Index

In [6]:
df_clean = df_clean.reset_index(drop=True)

### (g) Summary

In [7]:
print("\nAfter cleaning:", df_clean.shape)
print("Unique students:", df_clean['studId'].nunique())


After cleaning: (11130, 8)
Unique students: 799


### (h) Removed Rows

In [8]:
removed_rows = df.merge(df_clean, on='id', how='left', indicator=True)
removed_rows = removed_rows[removed_rows['_merge'] == 'left_only']

print("\nRemoved rows:", removed_rows.shape)
removed_rows.head(10)



Removed rows: (3465, 16)


,id,prompt_x,response_x,timestamp_x,studId_x,chatId_x,prompt_cn_x,response_cn_x,prompt_y,response_y,timestamp_y,studId_y,chatId_y,prompt_cn_y,response_cn_y,_merge
108,914,我想请你给我一点修改意见，特别是在人物的感受部分！“我和小文吵架了，心情很难过。”,你可以尝试更深入地描述人物的感情，比如他的心情是如何难过的？他的心疼到底像什么呢？他是如何表...,2024-03-15 08:56:00,223,127,True,True,NaN,NaN,NaT,NaN,NaN,NaN,NaN,left_only
109,915,请问我要如何描写他的心情如何难过呢,你可以考虑心难过的具体感觉，是像被厚厚的黑云覆盖，还是像被冷冷的雨水打湿？让我们设想一下，如...,2024-03-15 08:57:00,223,127,True,True,NaN,NaN,NaT,NaN,NaN,NaN,NaN,left_only
110,979,hello,你好，有什么可以帮助你的吗？我们是不是在写作中遇到了新的挑战呢?,2024-03-19 12:38:00,223,127,False,True,NaN,NaN,NaT,NaN,NaN,NaN,NaN,left_only
111,986,一天晚上十一点，小文还在玩电脑游戏，没有睡觉。妈妈看到了，很生气。句子同讯吗？,你的句子已经很好理解了。但是，我们是否可以更具体描述一下妈妈为什么会生气呢？例如，她是担心小...,2024-03-19 12:39:00,223,127,True,True,NaN,NaN,NaT,NaN,NaN,NaN,NaN,left_only
112,1000,你说有道理,太好了，你觉得有帮助。你想尝试再写出一段描述妈妈生气的情况吗？或者其他有关小文行为或思考的内容吗？,2024-03-19 12:43:00,223,127,True,True,NaN,NaN,NaT,NaN,NaN,NaN,NaN,left_only
113,1785,一天晚上十一点，小文还在玩电脑游戏，没有睡觉。妈妈看到了，很生气。请问可以帮助我修改这个句子...,我们可以考虑一下，小文看到妈妈生气后，心里是什么感觉呢？是害怕，是惊讶，还是有点失望自己？他...,2024-03-20 11:31:00,223,127,True,True,NaN,NaN,NaT,NaN,NaN,NaN,NaN,left_only
114,1796,小文会感到非常生气,好的，我们可以考虑一下小文生气的程度。他是简单地生气，还是愤怒到要气炸？他是火热的红脸，还是...,2024-03-20 11:35:00,223,127,True,True,NaN,NaN,NaT,NaN,NaN,NaN,NaN,left_only
115,1801,他闷闷不乐的去睡觉,"我们可以尝试更深层次地描述一下小文的动作。例如，他是怎么""闷闷不乐""地走向床的？是拖着沉重的...",2024-03-20 11:35:00,223,127,True,True,NaN,NaN,NaT,NaN,NaN,NaN,NaN,left_only
116,1814,他不愿意的去睡觉,"好的，那我们可以思考一下，他是怎么表现出""不愿意""去睡觉的呢？是瞪大眼睛看着妈妈，还是摇头抗...",2024-03-20 11:38:00,223,127,True,True,NaN,NaN,NaT,NaN,NaN,NaN,NaN,left_only
117,1826,因为他对妈妈发脾气，妈妈骂了他一顿，他就不愿意睡觉,那么小文是怎么表现他对妈妈的这种愤怒呢？他有没有什么特殊的眼神或动作？他是紧闭双拳，还是皱起...,2024-03-20 11:41:00,223,127,True,True,NaN,NaN,NaT,NaN,NaN,NaN,NaN,left_only


### (i) Final Rows

In [9]:
df_clean.head(10)

,id,prompt,response,timestamp,studId,chatId,prompt_cn,response_cn
0,20377,what should i eat today?,"As a teacher of Korean history and culture, I ...",2024-03-05 09:59:00,399,135,False,False
1,20384,recipe for kimchi,I can certainly guide you towards making your ...,2024-03-05 10:00:00,399,135,False,False
2,20456,basic greetings in korea,"Absolutely! In Korean culture, greetings are v...",2024-03-05 10:05:00,399,135,False,False
3,20528,basic words in korea,"Sure, here are some basic Korean words you can...",2024-03-05 10:06:00,399,135,False,False
4,20546,how to say you are so pretty in korea,"In Korean, if you want to tell someone that th...",2024-03-05 10:07:00,399,135,False,False
5,20680,what are the different needs and interest duri...,"During COVID-19, people's needs and interests ...",2024-03-05 10:34:00,399,135,False,False
6,20732,what are the unequal sharing of needs and inte...,Certainly. COVID-19 has brought many inequalit...,2024-03-05 10:40:00,399,135,False,False
7,24190,why is the differing needs and interests a gre...,The differing needs and interests in a diverse...,2024-11-05 21:30:00,399,135,False,False
8,24191,why is the unequal sharing of costs a great ch...,Unequal sharing of costs can lead to social in...,2024-11-05 22:10:00,399,135,False,False
9,24192,why is the unequal sharing of costs a great ch...,"The Singapore government, like any other, must...",2024-11-05 22:11:00,399,135,False,False


### (j) Save Output

In [10]:
df_clean.to_csv(OUTPUT_PROMPT, index=False, encoding="utf-8-sig")

removed_rows.to_csv("TeacherGAIA_removed_rows.csv", index=False, encoding="utf-8-sig")

print("Saved cleaned dataset:", OUTPUT_PROMPT)
print("Saved removed rows: TeacherGAIA_removed_rows.csv")

Saved cleaned dataset: TeacherGAIA_features_per_prompt.csv
Saved removed rows: TeacherGAIA_removed_rows.csv


## 3. Feature Engineering (Specific-Broad)

### 3.1 Feature 1: Query Length (Token Count)

This feature captures the **length of each prompt in terms of token count**, along with a **student-level average query length** to provide behavioral context.

---

#### Definition

For each prompt:

- `query_length` = number of whitespace-separated tokens in the prompt
- `avg_query_length` = average prompt length for each student (`studId`), computed across all their prompts

---

#### Implementation Details

- Tokenization is done using simple whitespace splitting:
  ```python
  df_feat['query_length'] = df_feat['prompt'].astype(str).str.split().apply(len)

In [11]:
df_feat = df_clean.copy()

# ensure clean tokenization (handles extra spaces robustly)
df_feat['query_length'] = df_feat['prompt'].astype(str).str.split().apply(len)

# student-level average query length
student_avg_len = (
    df_feat.groupby('studId')['query_length']
    .mean()
    .reset_index()
    .rename(columns={'query_length': 'avg_query_length'})
)

# merge back to per-prompt dataframe
df_feat = df_feat.merge(student_avg_len, on='studId', how='left')

# round off to 2 d.p
df_feat['avg_query_length'] = df_feat['avg_query_length'].round(2)

# remove prompt and response columns
df_feat = df_feat.drop(columns=['prompt_cn', 'response_cn'])
df_feat.head()

,id,prompt,response,timestamp,studId,chatId,query_length,avg_query_length
0,20377,what should i eat today?,"As a teacher of Korean history and culture, I ...",2024-03-05 09:59:00,399,135,5,9.9
1,20384,recipe for kimchi,I can certainly guide you towards making your ...,2024-03-05 10:00:00,399,135,3,9.9
2,20456,basic greetings in korea,"Absolutely! In Korean culture, greetings are v...",2024-03-05 10:05:00,399,135,4,9.9
3,20528,basic words in korea,"Sure, here are some basic Korean words you can...",2024-03-05 10:06:00,399,135,4,9.9
4,20546,how to say you are so pretty in korea,"In Korean, if you want to tell someone that th...",2024-03-05 10:07:00,399,135,9,9.9


### 3.2 Feature 2: Lexical Diversity (Shannon Entropy)

This feature measures the **diversity of vocabulary used by each student** using **Shannon Entropy**. It captures how varied or repetitive a student's language is across all their prompts.

---

#### Definition

For each student (`studId`):

- All tokens from all prompts are aggregated
- Shannon Entropy is computed over the full token distribution:

:contentReference[oaicite:0]{index=0}

where:
- \( p_i \) is the probability of token \( i \)
- \( H \) measures uncertainty / diversity of token usage

---

#### Implementation Details

##### 1. Tokenization
Each prompt is tokenized using a simple regex-based tokenizer:

```python
def simple_tokenize(text):
    if not isinstance(text, str):
        return []
    return re.findall(r\"[a-zA-Z0-9']+\", text.lower())

In [12]:
from collections import Counter
import math
import re

def shannon_entropy(tokens):
    if not tokens:
        return 0.0
    counts = Counter(tokens)
    total = sum(counts.values())
    return -sum((c / total) * math.log2(c / total) for c in counts.values())


def simple_tokenize(text):
    if not isinstance(text, str):
        return []
    return re.findall(r"[a-zA-Z0-9']+", text.lower())


df_feat = df_feat.copy()

# tokenize
df_feat['tokens'] = df_feat['prompt'].astype(str).apply(simple_tokenize)

# compute entropy per student (FORCE named column)
student_lexdiv = (
    df_feat.groupby('studId')['tokens']
    .apply(lambda lists: shannon_entropy([t for tl in lists for t in tl]))
    .reset_index(name='lexical_diversity')  
)

# merge
df_feat = df_feat.merge(student_lexdiv, on='studId', how='left')

# round safely
df_feat['lexical_diversity'] = df_feat['lexical_diversity'].round(2)

# cleanup
df_feat = df_feat.drop(columns=['tokens'], errors='ignore')

df_feat.head(20)

,id,prompt,response,timestamp,studId,chatId,query_length,avg_query_length,lexical_diversity
0,20377,what should i eat today?,"As a teacher of Korean history and culture, I ...",2024-03-05 09:59:00,399,135,5,9.90,5.26
1,20384,recipe for kimchi,I can certainly guide you towards making your ...,2024-03-05 10:00:00,399,135,3,9.90,5.26
2,20456,basic greetings in korea,"Absolutely! In Korean culture, greetings are v...",2024-03-05 10:05:00,399,135,4,9.90,5.26
3,20528,basic words in korea,"Sure, here are some basic Korean words you can...",2024-03-05 10:06:00,399,135,4,9.90,5.26
4,20546,how to say you are so pretty in korea,"In Korean, if you want to tell someone that th...",2024-03-05 10:07:00,399,135,9,9.90,5.26
5,20680,what are the different needs and interest duri...,"During COVID-19, people's needs and interests ...",2024-03-05 10:34:00,399,135,10,9.90,5.26
6,20732,what are the unequal sharing of needs and inte...,Certainly. COVID-19 has brought many inequalit...,2024-03-05 10:40:00,399,135,12,9.90,5.26
7,24190,why is the differing needs and interests a gre...,The differing needs and interests in a diverse...,2024-11-05 21:30:00,399,135,14,9.90,5.26
8,24191,why is the unequal sharing of costs a great ch...,Unequal sharing of costs can lead to social in...,2024-11-05 22:10:00,399,135,17,9.90,5.26
9,24192,why is the unequal sharing of costs a great ch...,"The Singapore government, like any other, must...",2024-11-05 22:11:00,399,135,21,9.90,5.26


### 3.3 Feature 3: Semantic Diversity (Sentence-Transformer Embeddings)

This feature measures the **semantic diversity of a student's prompts** using dense sentence embeddings from a pretrained transformer model. It captures how *meaningfully different* a student's prompts are from each other, beyond surface-level word usage.

---

#### Definition

For each student (`studId`):

- Each prompt is converted into a dense vector embedding using `SentenceTransformer`
- Pairwise Euclidean distances are computed between all embeddings
- Two statistics are extracted:

:contentReference[oaicite:0]{index=0}

- Mean distance → overall semantic spread  
- Variance of distances → consistency vs variability in semantic behavior

---

#### Implementation Details

##### 1. Load embedding model

```python
from sentence_transformers import SentenceTransformer

print('Loading sentence-transformer model (all-MiniLM-L6-v2)...')
st_model = SentenceTransformer('all-MiniLM-L6-v2')


In [13]:
from sentence_transformers import SentenceTransformer
from scipy.spatial.distance import pdist
import numpy as np
from tqdm import tqdm

# load model
print('Loading sentence-transformer model (all-MiniLM-L6-v2)...')
st_model = SentenceTransformer('all-MiniLM-L6-v2')

df_feat = df_feat.copy()

# encode unique prompts 
unique_prompts = df_feat['prompt'].unique().tolist()
print(f'Encoding {len(unique_prompts)} unique prompts...')

emb_list = st_model.encode(
    unique_prompts,
    batch_size=64,
    show_progress_bar=True
)

prompt_to_emb = {p: e for p, e in zip(unique_prompts, emb_list)}

# helper: pairwise stats 
def pairwise_stats(emb_matrix):
    """Return (mean, var) of pairwise Euclidean distances."""
    if len(emb_matrix) < 2:
        return 0.0, 0.0
    dists = pdist(emb_matrix, metric='euclidean')
    return float(np.mean(dists)), float(np.var(dists))

# compute student-level semantic diversity
print('Computing student-level semantic diversity...')

sem_rows = []

for stud, grp in tqdm(df_feat.groupby('studId'), desc='Students'):
    embs = np.array([prompt_to_emb[p] for p in grp['prompt']])
    m, v = pairwise_stats(embs)

    sem_rows.append({
        'studId': stud,
        'semantic_diversity_mean': m,
        'semantic_diversity_var': v
    })

sem_df = pd.DataFrame(sem_rows)

# merge back 
df_feat = df_feat.merge(sem_df, on='studId', how='left')

# round to 2dp (student-level features only) 
df_feat['semantic_diversity_mean'] = df_feat['semantic_diversity_mean'].round(2)
df_feat['semantic_diversity_var'] = df_feat['semantic_diversity_var'].round(2)

# sanity check 
df_feat.head(10)

Loading sentence-transformer model (all-MiniLM-L6-v2)...


Loading weights: 100%|██████████████████████| 103/103 [00:00<00:00, 6092.16it/s]


Encoding 9485 unique prompts...


Batches: 100%|████████████████████████████████| 149/149 [00:18<00:00,  7.96it/s]


Computing student-level semantic diversity...


Students: 100%|█████████████████████████████| 799/799 [00:00<00:00, 6323.15it/s]


,id,prompt,response,timestamp,studId,chatId,query_length,avg_query_length,lexical_diversity,semantic_diversity_mean,semantic_diversity_var
0,20377,what should i eat today?,"As a teacher of Korean history and culture, I ...",2024-03-05 09:59:00,399,135,5,9.9,5.26,1.25,0.05
1,20384,recipe for kimchi,I can certainly guide you towards making your ...,2024-03-05 10:00:00,399,135,3,9.9,5.26,1.25,0.05
2,20456,basic greetings in korea,"Absolutely! In Korean culture, greetings are v...",2024-03-05 10:05:00,399,135,4,9.9,5.26,1.25,0.05
3,20528,basic words in korea,"Sure, here are some basic Korean words you can...",2024-03-05 10:06:00,399,135,4,9.9,5.26,1.25,0.05
4,20546,how to say you are so pretty in korea,"In Korean, if you want to tell someone that th...",2024-03-05 10:07:00,399,135,9,9.9,5.26,1.25,0.05
5,20680,what are the different needs and interest duri...,"During COVID-19, people's needs and interests ...",2024-03-05 10:34:00,399,135,10,9.9,5.26,1.25,0.05
6,20732,what are the unequal sharing of needs and inte...,Certainly. COVID-19 has brought many inequalit...,2024-03-05 10:40:00,399,135,12,9.9,5.26,1.25,0.05
7,24190,why is the differing needs and interests a gre...,The differing needs and interests in a diverse...,2024-11-05 21:30:00,399,135,14,9.9,5.26,1.25,0.05
8,24191,why is the unequal sharing of costs a great ch...,Unequal sharing of costs can lead to social in...,2024-11-05 22:10:00,399,135,17,9.9,5.26,1.25,0.05
9,24192,why is the unequal sharing of costs a great ch...,"The Singapore government, like any other, must...",2024-11-05 22:11:00,399,135,21,9.9,5.26,1.25,0.05


### 3.4 Feature 4: Entity vs Category (Named Entity Recognition)

spaCy NER (`en_core_web_md`) extracts named entity types such as **PERSON, ORG, GPE, PRODUCT, EVENT**, etc.  
This feature measures how **diverse a user’s real-world entity focus is**, both at the student level and episode level, and how each query compares to the user’s overall behavior.

Unlike semantic diversity (which captures conceptual meaning), this feature focuses on structured real-world references extracted via named entities.

- Prompts mentioning specific entities (e.g., *“Einstein”, “Apple Inc”*) → more **specific**
- Prompts referring to general categories (e.g., *“scientists”, “tech companies”*) → more **broad**

---

#### Key Idea

We compute entropy-based diversity at multiple levels:

- **Student-level diversity** → how broadly a student explores different entity types across all prompts  
- **Episode-level diversity** → how diverse entity usage is within a chat session (`chatId`)  
- **Per-query ratios** → how each episode compares to the user’s overall behavior  
- **Episode-level ratios** → aggregated behavioral signal across queries in a session  

---

#### Output Columns

##### 1. Episode-level entity diversity

| Column | Meaning |
|--------|---------|
| `entity_category_diversity_ep` | Entropy of NER category labels within a chat session (episode-level diversity) |
| `specific_entity_diversity_ep` | Entropy of named entity text spans within a chat session |

---

##### 2. Student-level entity diversity

| Column | Meaning |
|--------|---------|
| `user_entity_category_diversity` | Entropy of entity category labels across all prompts of a student |
| `user_specific_entity_diversity` | Entropy of entity text spans across all prompts of a student |

---

##### 3. Per-query ratio features

These measure how much an episode deviates from the student’s overall behavior:

| Column | Meaning |
|--------|---------|
| `entity_category_ratio` | Episode category diversity ÷ user-level category diversity |
| `specific_entity_ratio` | Episode specific-entity diversity ÷ user-level specific-entity diversity |
| `category_entity_ratio` | Legacy alias of `entity_category_ratio` |

---

##### 4. Episode-level ratio features

Aggregated ratios across all queries within a chat session:

| Column | Meaning |
|--------|---------|
| `entity_category_ratio_episode` | Mean per-query category ratio within episode |
| `specific_entity_ratio_episode` | Mean per-query specific ratio within episode |
| `category_entity_ratio_episode` | Mean legacy ratio within episode |
| `overall_episode_ratio` | Average of all three episode-level ratios |

---

###  Interpretation

- **Low entropy / low ratio**
  → User focuses on a narrow set of entity types (e.g., only GPE such as “Singapore”, “Malaysia”)

- **High entropy / high ratio**
  → User explores diverse entity types (PERSON, ORG, PRODUCT, EVENT, etc.)

- **Ratio ≈ 1**
  → Episode behavior is similar to user’s typical behavior

- **Ratio < 1**
  → Episode is less diverse than user baseline

- **Ratio > 1**
  → Episode is more diverse than user baseline

- **Ratio = 0**
  → No entities detected or user-level entropy is zero


In [14]:
import spacy
import numpy as np
import pandas as pd
from collections import Counter
from tqdm import tqdm
import subprocess
import sys

# Load spaCy model
try:
    nlp_ner = spacy.load("en_core_web_md")
except OSError:
    subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_md"], check=True)
    nlp_ner = spacy.load("en_core_web_md")


# Entropy function

def ner_entropy(items):
    if not items:
        return 0.0
    counts = np.array(list(Counter(items).values()), dtype=float)
    p = counts / counts.sum()
    return float(-np.sum(p * np.log2(p + 1e-12)))


df_feat = df_feat.copy()

# Step 1: NER on unique prompts

unique_qs = df_feat["prompt"].unique().tolist()
print(f"Running NER on {len(unique_qs)} unique prompts...")

q_to_cats = {}
q_to_specs = {}

BATCH = 128

for i in tqdm(range(0, len(unique_qs), BATCH), desc="NER batches"):
    batch = unique_qs[i:i+BATCH]
    docs = nlp_ner.pipe(batch, disable=["tagger", "parser", "lemmatizer"])

    for q, doc in zip(batch, docs):
        q_to_cats[q] = [ent.label_ for ent in doc.ents]
        q_to_specs[q] = [ent.text for ent in doc.ents]


# Step 2: attach per-query NER
df_feat["_ner_cats"] = df_feat["prompt"].map(q_to_cats)
df_feat["_ner_specs"] = df_feat["prompt"].map(q_to_specs)


# Step 3: USER-LEVEL (studId) entropy
user_rows = []

for stud, grp in tqdm(df_feat.groupby("studId"), desc="User-level NER"):
    all_cats = [c for lst in grp["_ner_cats"] for c in lst]
    all_specs = [s for lst in grp["_ner_specs"] for s in lst]

    user_rows.append({
        "studId": stud,
        "user_entity_category_diversity": ner_entropy(all_cats),
        "user_specific_entity_diversity": ner_entropy(all_specs),
    })

user_df = pd.DataFrame(user_rows)
df_feat = df_feat.merge(user_df, on="studId", how="left")

# Step 4: EPISODE-LEVEL entropy (chatId = episode proxy)
ep_rows = []

for (stud, ep), grp in tqdm(df_feat.groupby(["studId", "chatId"]), desc="Episode-level NER"):
    all_cats = [c for lst in grp["_ner_cats"] for c in lst]
    all_specs = [s for lst in grp["_ner_specs"] for s in lst]

    ep_rows.append({
        "studId": stud,
        "chatId": ep,
        "entity_category_diversity_ep": ner_entropy(all_cats),
        "specific_entity_diversity_ep": ner_entropy(all_specs),
    })

ep_df = pd.DataFrame(ep_rows)
df_feat = df_feat.merge(ep_df, on=["studId", "chatId"], how="left")

# Step 5: PER-QUERY RATIOS
df_feat["entity_category_ratio"] = (
    df_feat["entity_category_diversity_ep"] /
    df_feat["user_entity_category_diversity"].replace(0, np.nan)
).fillna(0)

df_feat["specific_entity_ratio"] = (
    df_feat["specific_entity_diversity_ep"] /
    df_feat["user_specific_entity_diversity"].replace(0, np.nan)
).fillna(0)

df_feat["category_entity_ratio"] = df_feat["entity_category_ratio"]

# Step 6: EPISODE RATIOS

episode_ratio_df = (
    df_feat.groupby(["studId", "chatId"])[
        ["entity_category_ratio", "specific_entity_ratio", "category_entity_ratio"]
    ]
    .mean()
    .reset_index()
)

episode_ratio_df["entity_category_ratio_episode"] = episode_ratio_df["entity_category_ratio"]
episode_ratio_df["specific_entity_ratio_episode"] = episode_ratio_df["specific_entity_ratio"]
episode_ratio_df["category_entity_ratio_episode"] = episode_ratio_df["category_entity_ratio"]

episode_ratio_df["overall_episode_ratio"] = episode_ratio_df[
    [
        "entity_category_ratio_episode",
        "specific_entity_ratio_episode",
        "category_entity_ratio_episode",
    ]
].mean(axis=1)

df_feat = df_feat.merge(
    episode_ratio_df[
        [
            "studId",
            "chatId",
            "entity_category_ratio_episode",
            "specific_entity_ratio_episode",
            "category_entity_ratio_episode",
            "overall_episode_ratio",
        ]
    ],
    on=["studId", "chatId"],
    how="left",
)


# Step 7: cleanup

df_feat.drop(columns=["_ner_cats", "_ner_specs"], inplace=True, errors="ignore")



# Step 8: rounding

round_cols = [
    "entity_category_diversity_ep",
    "specific_entity_diversity_ep",
    "user_entity_category_diversity",
    "user_specific_entity_diversity",
    "entity_category_ratio",
    "specific_entity_ratio",
    "category_entity_ratio",
    "entity_category_ratio_episode",
    "specific_entity_ratio_episode",
    "category_entity_ratio_episode",
    "overall_episode_ratio",
]

df_feat[round_cols] = df_feat[round_cols].round(2)



# sanity check
df_feat.head(10)

Running NER on 9485 unique prompts...


Episode-level NER: 100%|███████████████████| 799/799 [00:00<00:00, 14863.72it/s]


,id,prompt,response,timestamp,studId,chatId,query_length,avg_query_length,lexical_diversity,semantic_diversity_mean,...,user_specific_entity_diversity,entity_category_diversity_ep,specific_entity_diversity_ep,entity_category_ratio,specific_entity_ratio,category_entity_ratio,entity_category_ratio_episode,specific_entity_ratio_episode,category_entity_ratio_episode,overall_episode_ratio
0,20377,what should i eat today?,"As a teacher of Korean history and culture, I ...",2024-03-05 09:59:00,399,135,5,9.9,5.26,1.25,...,1.84,0.59,1.84,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,20384,recipe for kimchi,I can certainly guide you towards making your ...,2024-03-05 10:00:00,399,135,3,9.9,5.26,1.25,...,1.84,0.59,1.84,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2,20456,basic greetings in korea,"Absolutely! In Korean culture, greetings are v...",2024-03-05 10:05:00,399,135,4,9.9,5.26,1.25,...,1.84,0.59,1.84,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3,20528,basic words in korea,"Sure, here are some basic Korean words you can...",2024-03-05 10:06:00,399,135,4,9.9,5.26,1.25,...,1.84,0.59,1.84,1.0,1.0,1.0,1.0,1.0,1.0,1.0
4,20546,how to say you are so pretty in korea,"In Korean, if you want to tell someone that th...",2024-03-05 10:07:00,399,135,9,9.9,5.26,1.25,...,1.84,0.59,1.84,1.0,1.0,1.0,1.0,1.0,1.0,1.0
5,20680,what are the different needs and interest duri...,"During COVID-19, people's needs and interests ...",2024-03-05 10:34:00,399,135,10,9.9,5.26,1.25,...,1.84,0.59,1.84,1.0,1.0,1.0,1.0,1.0,1.0,1.0
6,20732,what are the unequal sharing of needs and inte...,Certainly. COVID-19 has brought many inequalit...,2024-03-05 10:40:00,399,135,12,9.9,5.26,1.25,...,1.84,0.59,1.84,1.0,1.0,1.0,1.0,1.0,1.0,1.0
7,24190,why is the differing needs and interests a gre...,The differing needs and interests in a diverse...,2024-11-05 21:30:00,399,135,14,9.9,5.26,1.25,...,1.84,0.59,1.84,1.0,1.0,1.0,1.0,1.0,1.0,1.0
8,24191,why is the unequal sharing of costs a great ch...,Unequal sharing of costs can lead to social in...,2024-11-05 22:10:00,399,135,17,9.9,5.26,1.25,...,1.84,0.59,1.84,1.0,1.0,1.0,1.0,1.0,1.0,1.0
9,24192,why is the unequal sharing of costs a great ch...,"The Singapore government, like any other, must...",2024-11-05 22:11:00,399,135,21,9.9,5.26,1.25,...,1.84,0.59,1.84,1.0,1.0,1.0,1.0,1.0,1.0,1.0


### 3.5 Feature 5: Topic Diversity

Uses pretrained 100-topic clustering model (`google_query_2_model_100.pickle`)
to assign each prompt a topic ID. Shannon entropy over all topic IDs **per student** measures
how broadly a student's prompts span different topics.

> **Prerequisite:** Place `google_query_2_model_100.pickle` in the same folder as this notebook


Output columns:
| Column | Meaning |
|--------|---------|
| `topic` | Topic ID assigned to each prompt |
| `topic_diversity` | Shannon entropy of topic IDs across all of student's prompts |
| `topic_count` | Number of unique topic IDs across all of student's prompts |


In [15]:
import pickle
import os
import numpy as np
import pandas as pd
from collections import Counter
from tqdm import tqdm

# load topic model
if not os.path.exists(TOPIC_MODEL_PATH):
    raise FileNotFoundError(
        f"Topic model not found at '{TOPIC_MODEL_PATH}'.\n"
        "Obtain google_query_2_model_100.pickle from Serena Wan Jun Wei."
    )

print("Loading topic model...")
with open(TOPIC_MODEL_PATH, "rb") as f:
    topic_model = pickle.load(f)

if not hasattr(topic_model, "predict"):
    raise ValueError("Loaded model does not have .predict() method.")


# predict topics
print("Predicting topics...")

df_feat = df_feat.copy()

unique_ps = df_feat["prompt"].unique().tolist()

# reuse embeddings from semantic diversity step
emb_matrix = np.array([prompt_to_emb.get(p, np.zeros(384)) for p in unique_ps])

topics_pred = topic_model.predict(emb_matrix)
p_to_topic = {p: int(t) for p, t in zip(unique_ps, topics_pred)}

df_feat["topic"] = df_feat["prompt"].map(p_to_topic)


# topic entropy helper
def entropy(lst):
    if not lst:
        return 0.0
    arr = np.array(list(Counter(lst).values()), dtype=float)
    p = arr / arr.sum()
    return float(-np.sum(p * np.log2(p + 1e-12)))


# student-level topic features
topic_rows = []

for stud, grp in tqdm(df_feat.groupby("studId"), desc="Topic diversity"):
    tlist = grp["topic"].dropna().astype(int).tolist()

    topic_rows.append({
        "studId": stud,
        "topic_diversity": entropy(tlist),
        "topic_count": len(set(tlist)),
    })

topic_df = pd.DataFrame(topic_rows)

df_feat = df_feat.merge(topic_df, on="studId", how="left")

# rounding
df_feat["topic_diversity"] = df_feat["topic_diversity"].round(2)


df_feat.head(10)

Loading topic model...
Predicting topics...


Topic diversity: 100%|██████████████████████| 799/799 [00:00<00:00, 5922.60it/s]


,id,prompt,response,timestamp,studId,chatId,query_length,avg_query_length,lexical_diversity,semantic_diversity_mean,...,entity_category_ratio,specific_entity_ratio,category_entity_ratio,entity_category_ratio_episode,specific_entity_ratio_episode,category_entity_ratio_episode,overall_episode_ratio,topic,topic_diversity,topic_count
0,20377,what should i eat today?,"As a teacher of Korean history and culture, I ...",2024-03-05 09:59:00,399,135,5,9.9,5.26,1.25,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,84,2.45,6
1,20384,recipe for kimchi,I can certainly guide you towards making your ...,2024-03-05 10:00:00,399,135,3,9.9,5.26,1.25,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,57,2.45,6
2,20456,basic greetings in korea,"Absolutely! In Korean culture, greetings are v...",2024-03-05 10:05:00,399,135,4,9.9,5.26,1.25,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,70,2.45,6
3,20528,basic words in korea,"Sure, here are some basic Korean words you can...",2024-03-05 10:06:00,399,135,4,9.9,5.26,1.25,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,70,2.45,6
4,20546,how to say you are so pretty in korea,"In Korean, if you want to tell someone that th...",2024-03-05 10:07:00,399,135,9,9.9,5.26,1.25,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,70,2.45,6
5,20680,what are the different needs and interest duri...,"During COVID-19, people's needs and interests ...",2024-03-05 10:34:00,399,135,10,9.9,5.26,1.25,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,78,2.45,6
6,20732,what are the unequal sharing of needs and inte...,Certainly. COVID-19 has brought many inequalit...,2024-03-05 10:40:00,399,135,12,9.9,5.26,1.25,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,72,2.45,6
7,24190,why is the differing needs and interests a gre...,The differing needs and interests in a diverse...,2024-11-05 21:30:00,399,135,14,9.9,5.26,1.25,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,46,2.45,6
8,24191,why is the unequal sharing of costs a great ch...,Unequal sharing of costs can lead to social in...,2024-11-05 22:10:00,399,135,17,9.9,5.26,1.25,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,78,2.45,6
9,24192,why is the unequal sharing of costs a great ch...,"The Singapore government, like any other, must...",2024-11-05 22:11:00,399,135,21,9.9,5.26,1.25,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,46,2.45,6


### 3.6 Feature 6: Hypernym (WordNet Taxonomy Depth)

This feature measures **semantic specificity** of user queries using the WordNet taxonomy.

We compute the average `max_depth()` of WordNet synsets for content words in each prompt:
- Longer hypernym paths → more specific concepts
- Shorter hypernym paths → more general concepts

Unlike embedding-based features, this approach uses a **hand-crafted lexical ontology (WordNet)**, making it interpretable and grounded in linguistic structure.

---

#### Key Idea

We model specificity at three levels:

- **Prompt-level specificity** → how abstract or concrete a single query is  
- **Student-level baseline** → a user's average semantic depth across all prompts  
- **Relative deviation features** → how each query compares to the user's own behavior  

---

#### Output Columns

##### 1. Prompt-level features

| Column | Meaning |
|--------|---------|
| `hypernym_distance` | Average WordNet depth per prompt |
| `hypernym_normalised_score` | Min–max normalised depth within student (0–1 scale) |
| `hypernym_relative_z` | Z-score relative to student’s mean and variance |
| `hypernym_specificity` | `'specific'` (top 30% within student) or `'broad'` |

---

##### 2. Student-level features

| Column | Meaning |
|--------|---------|
| `student_hypernym_mean` | Average hypernym depth across all student prompts |
| `student_hypernym_std` | Standard deviation of hypernym depth |
| `student_hypernym_specificity_ratio` | Proportion of prompts classified as `'specific'` |

---

#### Interpretation

- **Low hypernym depth** → abstract/general queries (e.g., “technology”, “food”)  
- **High hypernym depth** → specific/concrete queries (e.g., “kimchi recipe steps”, “Einstein theory of relativity”)  

---

#### Behavioural Insight

- High `student_hypernym_mean` → user tends to ask **specific, grounded questions**  
- High `student_hypernym_specificity_ratio` → user frequently shifts into **high-detail querying behaviour**  
- High `hypernym_relative_z` → query is **more specific than user’s usual behaviour**  
- Low values → more **abstract or general exploratory behaviour**

In [16]:
import re
import numpy as np
import pandas as pd
from tqdm import tqdm
from nltk.corpus import wordnet as wn
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords as sw_corpus

# =========================
# Setup
# =========================
tqdm.pandas()

lemmatizer = WordNetLemmatizer()
stopwords = set(sw_corpus.words("english"))
hypernym_cache = {}

df_feat = df_feat.copy()

# =========================
# Preprocessing
# =========================
def preprocess_hypernym(text):
    text = str(text)

    text = re.sub(
        r'\b([A-Z0-9]+(?:\s+[A-Z0-9]+)+)\b',
        lambda m: m.group(0).replace(" ", "_"),
        text
    )

    text = text.lower()
    text = re.sub(r"[^a-z0-9_\s]", " ", text)

    tokens = [
        lemmatizer.lemmatize(t)
        for t in text.split()
        if t not in stopwords and not t.isdigit()
    ]

    return tokens

# =========================
# Hypernym distance
# =========================
def avg_hypernym_distance(query):
    tokens = preprocess_hypernym(query)

    if not tokens:
        return 0.0

    dists = []

    for tok in tokens:
        if tok not in hypernym_cache:
            synsets = wn.synsets(tok)

            if synsets:
                hyper_vals = [s.max_depth() for s in synsets]
                hypernym_cache[tok] = float(np.mean(hyper_vals))
            else:
                hypernym_cache[tok] = 0.0

        dists.append(hypernym_cache[tok])

    return float(np.mean(dists))

# =========================
# Step 1: raw feature
# =========================
print("Computing hypernym distances...")
df_feat["hypernym_distance"] = df_feat["prompt"].progress_apply(avg_hypernym_distance)

# =========================
# Step 2: student-level stats
# =========================
g = df_feat.groupby("studId")["hypernym_distance"]

df_feat["student_hypernym_mean"] = g.transform("mean")
df_feat["student_hypernym_std"]  = g.transform("std").fillna(0)

df_feat["student_hypernym_min"]  = g.transform("min")
df_feat["student_hypernym_max"]  = g.transform("max")

# =========================
# Step 3: normalised score (0–1 within student)
# =========================
df_feat["hypernym_normalised_score"] = (
    (df_feat["hypernym_distance"] - df_feat["student_hypernym_min"]) /
    (df_feat["student_hypernym_max"] - df_feat["student_hypernym_min"] + 1e-12)
)

# =========================
# Step 4: z-score
# =========================
df_feat["hypernym_relative_z"] = (
    (df_feat["hypernym_distance"] - df_feat["student_hypernym_mean"]) /
    (df_feat["student_hypernym_std"] + 1e-12)
)

# =========================
# Step 5: specificity label
# =========================
df_feat["hypernym_specificity"] = g.transform(
    lambda x: np.where(x > x.quantile(0.7), "specific", "broad")
)

# =========================
# Step 6: student-level ratio
# =========================
df_feat["student_hypernym_specificity_ratio"] = (
    df_feat.groupby("studId")["hypernym_specificity"]
    .transform(lambda s: (s == "specific").mean())
)

# =========================
# Step 7: CLEANUP
# =========================
df_feat.drop(
    columns=["student_hypernym_min", "student_hypernym_max"],
    inplace=True,
    errors="ignore"
)

# =========================
# Step 8: FINAL ROUNDING (ALL TO 2 DP)
# =========================
round_cols = [
    "hypernym_distance",
    "student_hypernym_mean",
    "student_hypernym_std",
    "hypernym_normalised_score",
    "hypernym_relative_z",
    "student_hypernym_specificity_ratio"
]

df_feat[round_cols] = df_feat[round_cols].round(2)

# (label stays string)
df_feat.head(10)

Computing hypernym distances...


100%|███████████████████████████████████| 11130/11130 [00:03<00:00, 2880.69it/s]


,id,prompt,response,timestamp,studId,chatId,query_length,avg_query_length,lexical_diversity,semantic_diversity_mean,...,topic,topic_diversity,topic_count,hypernym_distance,student_hypernym_mean,student_hypernym_std,hypernym_normalised_score,hypernym_relative_z,hypernym_specificity,student_hypernym_specificity_ratio
0,20377,what should i eat today?,"As a teacher of Korean history and culture, I ...",2024-03-05 09:59:00,399,135,5,9.9,5.26,1.25,...,84,2.45,6,2.42,3.07,0.91,0.19,-0.72,broad,0.3
1,20384,recipe for kimchi,I can certainly guide you towards making your ...,2024-03-05 10:00:00,399,135,3,9.9,5.26,1.25,...,57,2.45,6,2.50,3.07,0.91,0.23,-0.63,broad,0.3
2,20456,basic greetings in korea,"Absolutely! In Korean culture, greetings are v...",2024-03-05 10:05:00,399,135,4,9.9,5.26,1.25,...,70,2.45,6,3.20,3.07,0.91,0.49,0.14,broad,0.3
3,20528,basic words in korea,"Sure, here are some basic Korean words you can...",2024-03-05 10:06:00,399,135,4,9.9,5.26,1.25,...,70,2.45,6,4.52,3.07,0.91,1.00,1.59,specific,0.3
4,20546,how to say you are so pretty in korea,"In Korean, if you want to tell someone that th...",2024-03-05 10:07:00,399,135,9,9.9,5.26,1.25,...,70,2.45,6,2.72,3.07,0.91,0.31,-0.38,broad,0.3
5,20680,what are the different needs and interest duri...,"During COVID-19, people's needs and interests ...",2024-03-05 10:34:00,399,135,10,9.9,5.26,1.25,...,78,2.45,6,1.91,3.07,0.91,0.00,-1.27,broad,0.3
6,20732,what are the unequal sharing of needs and inte...,Certainly. COVID-19 has brought many inequalit...,2024-03-05 10:40:00,399,135,12,9.9,5.26,1.25,...,72,2.45,6,2.23,3.07,0.91,0.12,-0.92,broad,0.3
7,24190,why is the differing needs and interests a gre...,The differing needs and interests in a diverse...,2024-11-05 21:30:00,399,135,14,9.9,5.26,1.25,...,46,2.45,6,4.38,3.07,0.91,0.95,1.44,specific,0.3
8,24191,why is the unequal sharing of costs a great ch...,Unequal sharing of costs can lead to social in...,2024-11-05 22:10:00,399,135,17,9.9,5.26,1.25,...,78,2.45,6,2.93,3.07,0.91,0.39,-0.15,broad,0.3
9,24192,why is the unequal sharing of costs a great ch...,"The Singapore government, like any other, must...",2024-11-05 22:11:00,399,135,21,9.9,5.26,1.25,...,46,2.45,6,3.90,3.07,0.91,0.76,0.91,specific,0.3


### 3.7 Feature 7: Brysbaert Concreteness

Human-annotated concreteness ratings (Brysbaert et al., 2014) via `wordtangible`.
Scale: 1 (very abstract) → 5 (very concrete). Higher = more tangible, specific prompts.

Output columns:
| Column | Meaning |
|--------|---------|
| `avg_text_concreteness` | Mean concreteness per prompt (1–5, rounded to 2 dp) |
| `brysbaert_specificity` | `'concrete'` (≥3.0) or `'abstract'` (<3.0) |
| `student_brysbaert_concreteness` | Mean concreteness across all of student's prompts |
| `student_brysbaert_specificity_ratio` | Proportion of student's prompts labelled `'concrete'` |


In [17]:
from nltk.stem import WordNetLemmatizer as _WNLem
from nltk.corpus import stopwords as _sw_corp
from tqdm import tqdm
import numpy as np
import pandas as pd
import re

# =========================
# SAFE IMPORT (fallback included)
# =========================
try:
    from wordtangible import avg_text_concreteness as _wt_concreteness
except ModuleNotFoundError:
    print("WARNING: wordtangible not installed. Using fallback 0.0 scorer.")
    def _wt_concreteness(text):
        return 0.0

# =========================
# Setup
# =========================
tqdm.pandas()
df_feat = df_feat.copy()

_br_lemmatizer = _WNLem()
_br_stopwords  = set(_sw_corp.words('english'))
_br_cache      = {}

CONCRETENESS_THRESHOLD = 3.0

# =========================
# Preprocessing
# =========================
def _preprocess_brysbaert(query):
    query = str(query).lower()
    query = re.sub(r'[^a-z\s]', ' ', query)

    tokens = [
        _br_lemmatizer.lemmatize(t)
        for t in query.split()
        if t not in _br_stopwords and t.strip() != ""
    ]

    return ' '.join(tokens)

# =========================
# Concreteness function
# =========================
def compute_concreteness(query):
    clean = _preprocess_brysbaert(query)

    if clean in _br_cache:
        return _br_cache[clean]

    try:
        val = _wt_concreteness(clean)
        val = float(val) if val is not None else 0.0
    except Exception:
        val = 0.0

    _br_cache[clean] = val
    return val

# =========================
# Step 1: per-query feature
# =========================
print("Computing Brysbaert concreteness...")

df_feat["avg_text_concreteness"] = df_feat["prompt"].progress_apply(compute_concreteness)

# =========================
# Step 2: labeling
# =========================
df_feat["brysbaert_specificity"] = np.where(
    df_feat["avg_text_concreteness"] >= CONCRETENESS_THRESHOLD,
    "concrete",
    "abstract"
)

# =========================
# Step 3: student-level aggregates (FIXED)
# =========================
student_br = (
    df_feat.groupby("studId")
    .agg(
        student_brysbaert_concreteness=("avg_text_concreteness", "mean"),
        student_brysbaert_specificity_ratio=(
            "brysbaert_specificity",
            lambda s: (s == "concrete").mean()
        )
    )
    .reset_index()
)

# merge back
df_feat = df_feat.merge(student_br, on="studId", how="left")

# =========================
# Step 4: rounding (ALL SAFE)
# =========================
round_cols = [
    "avg_text_concreteness",
    "student_brysbaert_concreteness",
    "student_brysbaert_specificity_ratio"
]

df_feat[round_cols] = df_feat[round_cols].round(2)

# =========================
# sanity check
# =========================
df_feat[[
    "studId",
    "prompt",
    "avg_text_concreteness",
    "brysbaert_specificity",
    "student_brysbaert_concreteness",
    "student_brysbaert_specificity_ratio"
]].head(6)

Computing Brysbaert concreteness...


100%|███████████████████████████████████| 11130/11130 [00:02<00:00, 4461.40it/s]


,studId,prompt,avg_text_concreteness,brysbaert_specificity,student_brysbaert_concreteness,student_brysbaert_specificity_ratio
0,399,what should i eat today?,3.50,concrete,2.67,0.2
1,399,recipe for kimchi,4.43,concrete,2.67,0.2
2,399,basic greetings in korea,2.94,abstract,2.67,0.2
3,399,basic words in korea,2.91,abstract,2.67,0.2
4,399,how to say you are so pretty in korea,2.49,abstract,2.67,0.2
5,399,what are the different needs and interest duri...,1.82,abstract,2.67,0.2


### 3.8 Feature 8: Category Nouns Detection (Lexicon-Embedding Hybrid)

Flags prompts containing nouns that signal categorical thinking ("type", "kind", "group", etc.).
Final lexicon was built in the original pipeline via starter keywords + curated semantic neighbours.

Output columns:
| Column | Meaning |
|--------|---------|
| `Category_nouns` | `True`/`False` — any category noun present in the prompt |
| `Category_noun_words` | List of matched category words |


In [18]:
from sklearn.metrics.pairwise import cosine_similarity

# =========================
# 1. Starter lexicon
# =========================
CATEGORY_LEXICON_STARTER = sorted(set([
    'type', 'property', 'kind', 'group',
    'category', 'class', 'classification', 'variety', 'form', 'sort',
    'genre', 'species', 'subtype', 'subgroup', 'subcategory', 'subset',
    'division', 'section', 'branch', 'family', 'order', 'tier', 'level',
    'style', 'mode', 'pattern', 'collective', 'community', 'team',
    'organization', 'band', 'trio', 'cluster', 'set', 'collection',
    'series', 'array', 'range', 'scope', 'domain', 'field', 'area',
    'aspect', 'feature', 'characteristic', 'attribute', 'trait',
    'quality', 'nature', 'structure', 'element', 'component', 'factor',
    'example', 'instance', 'case', 'model', 'method', 'approach',
    'technique', 'mechanism', 'process', 'stage', 'phase', 'step'
]))

# =========================
# 2. Embedding model
# =========================
model = SentenceTransformer("all-MiniLM-L6-v2")

starter_emb = model.encode(CATEGORY_LEXICON_STARTER)

# =========================
# 3. Build vocabulary from dataset
# =========================
vocab = sorted(set(
    word.lower()
    for text in df_feat["prompt"].astype(str)
    for word in text.split()
))

vocab_emb = model.encode(vocab)

# =========================
# 4. Similarity computation
# =========================
sim_matrix = cosine_similarity(vocab_emb, starter_emb)

# =========================
# 5. Candidate generation (FOR HUMAN REVIEW)
# =========================
rows = []
for i, word in enumerate(vocab):
    best_idx = np.argmax(sim_matrix[i])
    best_score = sim_matrix[i][best_idx]
    best_seed = CATEGORY_LEXICON_STARTER[best_idx]

    rows.append([word, best_seed, best_score])

candidate_df = pd.DataFrame(rows, columns=[
    "word", "closest_seed", "similarity"
]).sort_values("similarity", ascending=False)

# SAVE candidates for manual inspection
candidate_df.to_csv("category_noun_candidates.csv", index=False)

# =========================
# 6. Human-chosen threshold
# =========================
SIM_THRESHOLD = 0.55  # adjust after inspecting CSV

expanded_lexicon = set(
    candidate_df[candidate_df["similarity"] >= SIM_THRESHOLD]["word"]
)

CATEGORY_LEXICON_FINAL = sorted(
    set(CATEGORY_LEXICON_STARTER).union(expanded_lexicon)
)

# =========================
# 7. Save all lexicons
# =========================
pd.Series(CATEGORY_LEXICON_STARTER).to_csv(
    "category_lexicon_starter.csv", index=False
)

pd.Series(list(expanded_lexicon)).to_csv(
    "category_lexicon_expanded.csv", index=False
)

pd.Series(CATEGORY_LEXICON_FINAL).to_csv(
    "category_lexicon_final.csv", index=False
)

# =========================
# 8. Build regex matcher from FINAL lexicon
# =========================
_lex_pattern = re.compile(
    r'\b(' + '|'.join(map(re.escape, CATEGORY_LEXICON_FINAL)) + r')\b',
    re.IGNORECASE
)

# =========================
# 9. Detection function
# =========================
def detect_category_nouns(query):
    matches = _lex_pattern.findall(str(query).lower())
    uniq = list(dict.fromkeys(m.lower() for m in matches))
    return bool(uniq), uniq

# =========================
# 10. Apply to dataframe
# =========================
tqdm.pandas()
print("Detecting category nouns...")

_res = df_feat["prompt"].progress_apply(detect_category_nouns)

df_feat["Category_nouns"] = _res.apply(lambda x: x[0])
df_feat["Category_noun_words"] = _res.apply(lambda x: x[1])

# =========================
# 11. Summary stats
# =========================
n_cat = df_feat["Category_nouns"].sum()

print(
    f"Prompts with category nouns: {n_cat} / {len(df_feat)} "
    f"({n_cat/len(df_feat)*100:.1f}%)"
)

# =========================
# 12. Save final dataset
# =========================

output_path = "prompt_specific_broad.csv"

df_feat.to_csv(output_path, index=False)

print(f"Final dataset saved to: {output_path}")


Loading weights: 100%|██████████████████████| 103/103 [00:00<00:00, 3451.61it/s]


Detecting category nouns...


100%|███████████████████████████████████| 11130/11130 [00:01<00:00, 6347.53it/s]


Prompts with category nouns: 3357 / 11130 (30.2%)
Final dataset saved to: prompt_specific_broad.csv


In [19]:
lexicon_starter = pd.read_csv("category_lexicon_starter.csv")
print(lexicon_starter)

            0
0    approach
1        area
2       array
3      aspect
4   attribute
..        ...
60       tier
61      trait
62       trio
63       type
64    variety

[65 rows x 1 columns]


In [20]:
# shows cosine similarity
noun_candidates = pd.read_csv("category_lexicon_starter.csv")
print(noun_candidates)

            0
0    approach
1        area
2       array
3      aspect
4   attribute
..        ...
60       tier
61      trait
62       trio
63       type
64    variety

[65 rows x 1 columns]


In [21]:
lexicon_expanded = pd.read_csv("category_lexicon_expanded.csv")
print(lexicon_expanded)

              0
0         step.
1    households
2          ngos
3        varied
4         clans
..          ...
540        step
541      cases,
542   distance.
543    stylefor
544        sort

[545 rows x 1 columns]


In [22]:
lexicon_final = pd.read_csv("category_lexicon_final.csv")
print(lexicon_final)

              0
0    "community
1        "three
2       "three!
3    'community
4       (factor
..          ...
558       ytype
559        zone
560       zones
561      zones,
562    “factor,

[563 rows x 1 columns]


## 4. Feature Engineering (Surface-Deep)

### 4.1 Feature 9: Question Detection (Regex)

This feature identifies whether a user prompt is framed as a direct question. Question-like queries typically reflect surface-level information seeking, where users request explicit answers rather than engaging in exploratory or conceptual reasoning.

We flag prompts as **question-type** if they either:
- Begin with an interrogative or auxiliary verb (e.g., *what, who, where, how, is, can, do*, etc.), or  
- End with a question mark (`?`)

This is implemented using a rule-based regular expression that captures common English question structures.

---

#### Output Columns

| Column | Meaning |
|--------|--------|
| `is_question` | `"question"` if the prompt matches the regex pattern, otherwise `"not_question"` |
| `user_question_rate` | Proportion of a student’s prompts classified as `"question"` |

---

#### Method Summary

**(i) Regex-Based Classification**  
Each prompt is converted to lowercase and matched against a predefined regex pattern that detects:
- WH-questions (what, why, how, etc.)
- Auxiliary verb questions (is, can, do, will, etc.)
- Terminal question punctuation (`?`)

**(ii) Label Assignment**
- `"question"` → Prompt matches the regex pattern  
- `"not_question"` → Prompt does not match  

**(iii) User-Level Aggregation**  
For each student (`studId`), we compute `user_question_rate`, defined as:

\[
user\_question\_rate = \frac{\text{number of question prompts}}{\text{total prompts}}
\]

This measures the proportion of a user's prompts that are question-based.

---

#### Significance

- **Surface-level search behaviour**: High question frequency indicates direct, fact-seeking queries.
- **Exploratory depth contrast**: Lower question frequency suggests more descriptive or reasoning-based prompts.
- **User profiling**: Enables comparison of users based on query style (question-driven vs exploratory).tarting with a question word or ending with `?`.


In [112]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm

tqdm.pandas()

# =========================
# 1. Create surface-deep dataframe
# =========================
df_sd = df_clean.copy()

# FIX: actually drop columns
df_sd = df_sd.drop(columns=['prompt_cn', 'response_cn'], errors='ignore')

# =========================
# 2. Question regex
# =========================
QUESTION_PATTERN = re.compile(
    r'(?i)^\s*(what|who|where|when|why|how|which|whose|whom|'
    r'is|are|was|were|do|does|did|can|could|will|would|shall|should|'
    r'has|have|had|may|might|must|need|dare|used)\b|'
    r'\?\s*$'
)

# =========================
# 3. Classification function
# =========================
def classify_question(query):
    q = str(query).strip()
    return "question" if QUESTION_PATTERN.search(q) else "not_question"

# =========================
# 4. Apply regex
# =========================
print("Running question detection...")

df_sd["is_question"] = df_sd["prompt"].progress_apply(classify_question)

# =========================
# 5. User-level question rate
# =========================
user_question_rate = (
    df_sd.groupby("studId")["is_question"]
    .apply(lambda s: (s == "question").mean())
    .reset_index()
    .rename(columns={"is_question": "user_question_rate"})
)

df_sd = df_sd.merge(user_question_rate, on="studId", how="left")

# =========================
# 6. Round metric
# =========================
df_sd["user_question_rate"] = df_sd["user_question_rate"].round(4)

# =========================
# 7. Sanity check
# =========================
df_sd.head(10)

Running question detection...


100%|█████████████████████████████████| 11130/11130 [00:00<00:00, 371593.94it/s]


,id,prompt,response,timestamp,studId,chatId,is_question,user_question_rate
0,20377,what should i eat today?,"As a teacher of Korean history and culture, I ...",2024-03-05 09:59:00,399,135,question,0.7
1,20384,recipe for kimchi,I can certainly guide you towards making your ...,2024-03-05 10:00:00,399,135,not_question,0.7
2,20456,basic greetings in korea,"Absolutely! In Korean culture, greetings are v...",2024-03-05 10:05:00,399,135,not_question,0.7
3,20528,basic words in korea,"Sure, here are some basic Korean words you can...",2024-03-05 10:06:00,399,135,not_question,0.7
4,20546,how to say you are so pretty in korea,"In Korean, if you want to tell someone that th...",2024-03-05 10:07:00,399,135,question,0.7
5,20680,what are the different needs and interest duri...,"During COVID-19, people's needs and interests ...",2024-03-05 10:34:00,399,135,question,0.7
6,20732,what are the unequal sharing of needs and inte...,Certainly. COVID-19 has brought many inequalit...,2024-03-05 10:40:00,399,135,question,0.7
7,24190,why is the differing needs and interests a gre...,The differing needs and interests in a diverse...,2024-11-05 21:30:00,399,135,question,0.7
8,24191,why is the unequal sharing of costs a great ch...,Unequal sharing of costs can lead to social in...,2024-11-05 22:10:00,399,135,question,0.7
9,24192,why is the unequal sharing of costs a great ch...,"The Singapore government, like any other, must...",2024-11-05 22:11:00,399,135,question,0.7


### 4.2 Feature 10: Closure Detection (Lexicon-Embedding Hybrid)

This feature identifies prompts that seek **definitive, closed-form answers**, where the user expects a precise explanation, definition, or direct factual response rather than open-ended exploration.

Such prompts typically include terms like *definition*, *meaning*, *synonym*, or phrases such as *what is* and *what does*, which indicate an intent to obtain specific and conclusive information.

---

#### Lexicon–Embedding Hybrid Strategy

To improve coverage while maintaining precision, a hybrid lexicon construction approach is used:

1. **Starter Lexicon Construction**  
   A manually curated list of high-confidence closure-related keywords is defined (e.g., definition, meaning, explain, synonym, what is).

2. **Embedding-Based Expansion**  
   Words from the dataset vocabulary are embedded using a pretrained sentence embedding model. Cosine similarity is computed between vocabulary words and starter lexicon terms to identify semantically related candidates.

3. **Threshold Selection (Human Review)**  
   Candidate words are ranked by similarity score. A similarity threshold is selected after manual inspection to filter noise and retain meaningful additions.

4. **Final Lexicon Formation**  
   The final lexicon is constructed as:
   - Starter lexicon (high-precision seed terms)
   - + Expanded lexicon (validated embedding-based candidates)

This ensures a balance between **precision (rule-based detection)** and **recall (semantic expansion)**.

---

#### Output Features

| Column | Meaning |
|--------|--------|
| closure | Boolean flag indicating whether a prompt contains closure-related lexicon terms |
| closure_words | List of matched closure-related keywords found in the prompt |
| user_closure_rate | Proportion of a user’s prompts classified as closure-seeking |

---

#### Significance

- **Surface-level information seeking**: Captures prompts requesting direct answers or definitions.
- **Intent classification**: Distinguishes definitional queries from exploratory or analytical prompts.
- **Behavioral profiling**: `user_closure_rate` measures a user’s tendency toward closure-seeking behavior.

In [114]:
import re
import pandas as pd
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

tqdm.pandas()

df_sd = df_sd.copy()

# =========================
# 1. Starter lexicon
# =========================
CLOSURE_LEXICON_STARTER = sorted(set([
    'definition', 'meaning', 'define', 'synonym', 'antonym', 'opposite',
    'explain', 'explanation', 'overview', 'summary', 'answer', 'conclusion',
    'result', 'solution', 'final', 'difference between', 'compare', 'versus',
    'vs', 'abbreviation', 'acronym', 'term', 'terminology',
    'what is', 'what are', 'what does', 'what do', 'what was', 'what were'
]))

# =========================
# 2. Embedding model
# =========================
model = SentenceTransformer("all-MiniLM-L6-v2")

starter_emb = model.encode(CLOSURE_LEXICON_STARTER)

# =========================
# 3. Build vocab from df_sd
# =========================
vocab = sorted(set(
    word.lower()
    for text in df_sd["prompt"].astype(str)
    for word in text.split()
))

vocab_emb = model.encode(vocab)

# =========================
# 4. Cosine similarity
# =========================
sim_matrix = cosine_similarity(vocab_emb, starter_emb)

# =========================
# 5. Candidate table (SAVE ONLY THIS, like your noun pipeline)
# =========================
rows = []
for i, word in enumerate(vocab):
    best_idx = np.argmax(sim_matrix[i])
    best_score = sim_matrix[i][best_idx]
    best_seed = CLOSURE_LEXICON_STARTER[best_idx]

    rows.append([word, best_seed, best_score])

closure_candidates = pd.DataFrame(rows, columns=[
    "word", "closest_seed", "similarity"
]).sort_values("similarity", ascending=False)

closure_candidates.to_csv("closure_candidates.csv", index=False)

# =========================
# 6. Final lexicon threshold
# =========================
SIM_THRESHOLD = 0.55

expanded_closure = set(
    closure_candidates[closure_candidates["similarity"] >= SIM_THRESHOLD]["word"]
)

CLOSURE_LEXICON_FINAL = sorted(
    set(CLOSURE_LEXICON_STARTER).union(expanded_closure)
)

# =========================
# 7. Save lexicons ONLY (like your noun pipeline)
# =========================
pd.Series(CLOSURE_LEXICON_STARTER).to_csv(
    "closure_lexicon_starter.csv", index=False
)

pd.Series(list(expanded_closure)).to_csv(
    "closure_lexicon_expanded.csv", index=False
)

pd.Series(CLOSURE_LEXICON_FINAL).to_csv(
    "closure_lexicon_final.csv", index=False
)

# =========================
# 8. Regex matcher (for analysis only, not saved df)
# =========================
closure_pattern = re.compile(
    r'\b(' + '|'.join(map(re.escape, CLOSURE_LEXICON_FINAL)) + r')\b',
    re.IGNORECASE
)

def detect_closure(query):
    matches = closure_pattern.findall(str(query).lower())
    uniq = list(dict.fromkeys(m.lower() for m in matches))
    return bool(uniq), uniq

# =========================
# 9. Apply ONLY feature columns (no dataframe export)
# =========================
print("Detecting closure prompts...")

_res = df_sd["prompt"].progress_apply(detect_closure)

df_sd["closure"] = _res.apply(lambda x: x[0])
df_sd["closure_words"] = _res.apply(lambda x: x[1])

# =========================
# 10. User-level metric
# =========================
user_closure_rate = (
    df_sd.groupby("studId")["closure"]
    .mean()
    .reset_index()
    .rename(columns={"closure": "user_closure_rate"})
)

df_sd = df_sd.merge(user_closure_rate, on="studId", how="left")

# =========================
# 11. Summary ONLY
# =========================
n_closure = df_sd["closure"].sum()
print(f"Closure prompts: {n_closure} / {len(df_sd)} ({n_closure/len(df_sd)*100:.1f}%)")

# =========================
# 12. Inspect outputs like your noun pipeline
# =========================
print("\nStarter lexicon:")
print(pd.read_csv("closure_lexicon_starter.csv"))

print("\nExpanded lexicon:")
print(pd.read_csv("closure_lexicon_expanded.csv"))

print("\nFinal lexicon:")
print(pd.read_csv("closure_lexicon_final.csv"))

print("\nCandidate similarity table:")
print(closure_candidates.head(10))

Loading weights: 100%|██████████████████████| 103/103 [00:00<00:00, 3831.43it/s]


Detecting closure prompts...


100%|███████████████████████████████████| 11130/11130 [00:03<00:00, 3366.10it/s]

Closure prompts: 4732 / 11130 (42.5%)

Starter lexicon:
                     0
0         abbreviation
1              acronym
2               answer
3              antonym
4              compare
5           conclusion
6               define
7           definition
8   difference between
9              explain
10         explanation
11               final
12             meaning
13            opposite
14            overview
15              result
16            solution
17             summary
18             synonym
19                term
20         terminology
21              versus
22                  vs
23            what are
24             what do
25           what does
26             what is
27            what was
28           what were

Expanded lexicon:
                  0
0         different
1        difference
2           comment
3             final
4      explanation?
..              ...
235         solving
236  were.suddenly,
237       summaries
238         answer.
239       solut

### 4.3 Feature 11: Comparative Detection (Lexicon-Embedding Hybrid)

This feature identifies prompts containing **comparative reasoning or evaluative language**, where the user compares alternatives, contrasts concepts, or evaluates relative advantages and disadvantages.

Comparative prompts often indicate **deeper analytical thinking**, as users are not merely requesting factual information but are actively evaluating relationships, trade-offs, or differences between entities.

Examples include phrases such as *better than*, *pros and cons*, *versus*, *compare*, and *difference between*.

---

#### Lexicon–Embedding Hybrid Strategy

To improve robustness and semantic coverage, a hybrid lexicon construction approach is adopted:

1. **Starter Lexicon Construction**  
   A manually curated set of high-confidence comparative keywords is defined (e.g., better, versus, compare, advantage, disadvantage, pros, cons).

2. **Embedding-Based Expansion**  
   Vocabulary words from the dataset are embedded using a pretrained sentence embedding model. Cosine similarity is computed between vocabulary terms and starter lexicon words to identify semantically related candidates.

3. **Threshold Selection (Human Review)**  
   Candidate words are ranked according to cosine similarity scores. A similarity threshold is selected after manual inspection to remove noisy or irrelevant matches.

4. **Final Lexicon Formation**  
   The final comparative lexicon is formed as the union of:
   - Starter lexicon (high-precision seed terms)
   - Expanded lexicon (embedding-derived semantic neighbours)

This balances:
- **Precision** through rule-based seed terms
- **Recall** through semantic expansion

---

#### Output Features

| Column | Meaning |
|--------|--------|
| comparative | Boolean flag indicating whether a prompt contains comparative language |
| comparative_words | List of matched comparative keywords detected in the prompt |
| user_comparative_rate | Proportion of a user’s prompts classified as comparative |

---

#### Significance

- **Analytical reasoning indicator**: Comparative prompts often reflect deeper evaluation and critical thinking.
- **Exploratory behavior detection**: Users comparing alternatives may be engaging in broader exploration rather than direct answer retrieval.
- **Behavioral profiling**: `user_comparative_rate` captures an individual user’s tendency toward evaluative and contrastive search behavior.

In [116]:
# ---------------------------------------------------------
# 1. Starter comparative lexicon
# ---------------------------------------------------------
COMPARATIVE_LEXICON_STARTER = sorted(set([
    'better', 'worse', 'more', 'less',
    'higher', 'lower', 'faster', 'slower',
    'greater', 'smaller', 'larger',
    'easier', 'harder', 'cheaper', 'expensive',
    'versus', 'vs', 'compared to',
    'in comparison', 'differ', 'difference',
    'advantage', 'disadvantage',
    'pros', 'cons',
    'similar', 'unlike',
    'contrast', 'whereas', 'while',
    'although', 'however', 'despite',
    'on the other hand',
    'alternatively', 'instead'
]))

# ---------------------------------------------------------
# 2. Load embedding model
# ---------------------------------------------------------
print("Loading embedding model...")

model = SentenceTransformer("all-MiniLM-L6-v2")

starter_emb = model.encode(COMPARATIVE_LEXICON_STARTER)

# ---------------------------------------------------------
# 3. Build vocabulary from prompts
# ---------------------------------------------------------
print("Building vocabulary from prompts...")

vocab = sorted(set(
    word.lower()
    for text in df_sd["prompt"].astype(str)
    for word in text.split()
))

vocab_emb = model.encode(vocab)

# ---------------------------------------------------------
# 4. Cosine similarity computation
# ---------------------------------------------------------
print("Computing cosine similarity...")

sim_matrix = cosine_similarity(vocab_emb, starter_emb)

# ---------------------------------------------------------
# 5. Candidate neighbour generation
# ---------------------------------------------------------
rows = []

for i, word in enumerate(vocab):

    best_idx = np.argmax(sim_matrix[i])

    best_score = sim_matrix[i][best_idx]

    best_seed = COMPARATIVE_LEXICON_STARTER[best_idx]

    rows.append([
        word,
        best_seed,
        round(float(best_score), 4)
    ])

comparative_candidate_df = pd.DataFrame(
    rows,
    columns=[
        "word",
        "closest_seed",
        "similarity"
    ]
).sort_values(
    "similarity",
    ascending=False
)

# ---------------------------------------------------------
# 6. Save candidate neighbours
# ---------------------------------------------------------
comparative_candidate_df.to_csv(
    "comparative_candidates.csv",
    index=False
)

# ---------------------------------------------------------
# 7. Human-selected similarity threshold
# ---------------------------------------------------------
SIM_THRESHOLD = 0.55

expanded_lexicon = set(
    comparative_candidate_df[
        comparative_candidate_df["similarity"] >= SIM_THRESHOLD
    ]["word"]
)

# ---------------------------------------------------------
# 8. Final comparative lexicon
# ---------------------------------------------------------
COMPARATIVE_LEXICON_FINAL = sorted(
    set(COMPARATIVE_LEXICON_STARTER).union(expanded_lexicon)
)

# ---------------------------------------------------------
# 9. Save lexicons
# ---------------------------------------------------------
pd.Series(COMPARATIVE_LEXICON_STARTER).to_csv(
    "comparative_lexicon_starter.csv",
    index=False
)

pd.Series(sorted(expanded_lexicon)).to_csv(
    "comparative_lexicon_expanded.csv",
    index=False
)

pd.Series(COMPARATIVE_LEXICON_FINAL).to_csv(
    "comparative_lexicon_final.csv",
    index=False
)

# ---------------------------------------------------------
# 10. Regex matcher using FINAL lexicon
# ---------------------------------------------------------
_comp_pattern = re.compile(
    r'(?i)\b(' +
    '|'.join(re.escape(k) for k in COMPARATIVE_LEXICON_FINAL) +
    r')\b'
)

# ---------------------------------------------------------
# 11. Comparative classifier
# ---------------------------------------------------------
def classify_comparative(query):

    matches = _comp_pattern.findall(str(query))

    uniq = list(dict.fromkeys(
        m.lower() for m in matches
    ))

    if uniq:
        return pd.Series([
            'comparative',
            uniq
        ])

    return pd.Series([
        'not_comparative',
        []
    ])

# ---------------------------------------------------------
# 12. Apply comparative detection
# ---------------------------------------------------------
tqdm.pandas()

print("Detecting comparative prompts...")

df_sd[['comparative', 'comparative_words']] = (
    df_sd['prompt']
    .progress_apply(classify_comparative)
)

# ---------------------------------------------------------
# 13. Student-level comparative rate
# ---------------------------------------------------------
user_comp_rate = (
    df_sd.groupby('studId')['comparative']
    .apply(lambda s: (s == 'comparative').mean())
    .reset_index()
    .rename(columns={
        'comparative': 'user_comparative_rate'
    })
)

# remove old column if rerunning notebook
if 'user_comparative_rate' in df_sd.columns:
    df_sd = df_sd.drop(
        columns=['user_comparative_rate']
    )

# merge back
df_sd = df_sd.merge(
    user_comp_rate,
    on='studId',
    how='left'
)

# ---------------------------------------------------------
# 14. Summary statistics
# ---------------------------------------------------------
n_comp = (df_sd['comparative'] == 'comparative').sum()

print(
    f"Comparative prompts: {n_comp} / {len(df_sd)} "
    f"({n_comp/len(df_sd)*100:.1f}%)"
)

# ---------------------------------------------------------
# 15. Display samples
# ---------------------------------------------------------
print("\nComparative detection sample:\n")

print(
    df_sd[
        [
            'studId',
            'prompt',
            'comparative',
            'comparative_words',
            'user_comparative_rate'
        ]
    ].head(10)
)

# ---------------------------------------------------------
# 16. Display saved lexicons
# ---------------------------------------------------------
print("\n=== STARTER LEXICON ===")
comparative_starter = pd.read_csv(
    "comparative_lexicon_starter.csv"
)
print(comparative_starter)

print("\n=== CANDIDATE NEIGHBOURS ===")
comparative_candidates = pd.read_csv(
    "comparative_candidates.csv"
)
print(comparative_candidates)

print("\n=== EXPANDED LEXICON ===")
comparative_expanded = pd.read_csv(
    "comparative_lexicon_expanded.csv"
)
print(comparative_expanded)

print("\n=== FINAL LEXICON ===")
comparative_final = pd.read_csv(
    "comparative_lexicon_final.csv"
)
print(comparative_final)

Loading embedding model...


Loading weights: 100%|██████████████████████| 103/103 [00:00<00:00, 5527.86it/s]


Building vocabulary from prompts...
Computing cosine similarity...
Detecting comparative prompts...


100%|███████████████████████████████████| 11130/11130 [00:02<00:00, 5353.53it/s]

Comparative prompts: 5005 / 11130 (45.0%)

Comparative detection sample:

   studId                                             prompt      comparative  \
0     399                           what should i eat today?  not_comparative   
1     399                                  recipe for kimchi  not_comparative   
2     399                           basic greetings in korea  not_comparative   
3     399                               basic words in korea  not_comparative   
4     399              how to say you are so pretty in korea  not_comparative   
5     399  what are the different needs and interest duri...      comparative   
6     399  what are the unequal sharing of needs and inte...      comparative   
7     399  why is the differing needs and interests a gre...      comparative   
8     399  why is the unequal sharing of costs a great ch...      comparative   
9     399  why is the unequal sharing of costs a great ch...      comparative   

             comparative_words  us

### 4.4 Feature 12: Reformulation (Levenshtein Distance)

Edit distance between **consecutive prompts from the same student** (ordered by row position,
which reflects the original data order). Measures how much a student reformulates their prompt.

> In the original Google Search pipeline this was computed within query episodes.
> Here it is computed across consecutive prompts per student (no session boundaries).


In [118]:
# ----------------------------------------------------------
# Install RapidFuzz (fast Levenshtein implementation)
# ----------------------------------------------------------
try:
    from rapidfuzz import distance
except:
    import sys
    import subprocess

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "rapidfuzz",
        "--quiet"
    ])

    from rapidfuzz import distance

# ==========================================================
# DESCRIPTION
# ==========================================================
# Reformulation measures how much a student modifies
# their prompt from one query to the next.
#
# Unlike most other features, this feature preserves
# the ORIGINAL sequential prompt order.
#
# Features generated:
#
# 1. reformulation_levenshtein
#    -> raw edit distance
#
# 2. reformulation_levenshtein_norm
#    -> normalized edit distance (0–1)
#
# 3. user_reformulation_mean
#    -> student's average normalized reformulation
#
# 4. user_reformulation_std
#    -> student's reformulation variability
# ==========================================================

# ----------------------------------------------------------
# Safety copy
# ----------------------------------------------------------
df_sd = df_sd.copy()

# ----------------------------------------------------------
# Preserve original dataset order
# ----------------------------------------------------------
df_sd = (
    df_sd
    .reset_index(drop=False)
    .rename(columns={"index": "_orig_order"})
)

# sort sequentially within each student
df_sd = (
    df_sd
    .sort_values(["studId", "_orig_order"])
    .reset_index(drop=True)
)

# ----------------------------------------------------------
# Safe Levenshtein helper
# ----------------------------------------------------------
def compute_lev(prev_q, cur_q):

    # first query
    if prev_q is None or pd.isna(prev_q):
        return 0, 0.0

    # missing current query
    if cur_q is None or pd.isna(cur_q):
        return 0, 0.0

    prev_q_str = str(prev_q)
    cur_q_str  = str(cur_q)

    # raw edit distance
    lev = distance.Levenshtein.distance(
        prev_q_str,
        cur_q_str
    )

    # normalized distance
    lev_norm = lev / max(
        len(prev_q_str),
        len(cur_q_str),
        1
    )

    lev_norm = round(float(lev_norm), 2)

    return lev, lev_norm

# ----------------------------------------------------------
# Sequential reformulation computation
# ----------------------------------------------------------
print("Computing reformulation distances...")

last_query_by_student = {}

levs_orig = []
levs_norm = []

for row in tqdm(
    df_sd.itertuples(index=False),
    total=len(df_sd)
):

    student = row.studId
    cur_q   = row.prompt

    prev_q = last_query_by_student.get(
        student,
        None
    )

    lev, lev_norm = compute_lev(
        prev_q,
        cur_q
    )

    levs_orig.append(lev)
    levs_norm.append(lev_norm)

    # update previous query
    last_query_by_student[student] = cur_q

# ----------------------------------------------------------
# Add reformulation columns
# ----------------------------------------------------------
df_sd["reformulation_levenshtein"] = (
    pd.Series(levs_orig)
    .fillna(0)
    .astype(int)
)

df_sd["reformulation_levenshtein_norm"] = (
    pd.Series(levs_norm)
    .fillna(0)
    .round(2)
)

# ----------------------------------------------------------
# Student-level reformulation tendency
# ----------------------------------------------------------
student_reformulation = (
    df_sd.groupby("studId")
    .agg(
        user_reformulation_mean=(
            "reformulation_levenshtein_norm",
            "mean"
        ),
        user_reformulation_std=(
            "reformulation_levenshtein_norm",
            "std"
        )
    )
    .reset_index()
)

# remove old columns if rerunning notebook
for col in [
    "user_reformulation_mean",
    "user_reformulation_std"
]:
    if col in df_sd.columns:
        df_sd = df_sd.drop(columns=[col])

# merge aggregates back
df_sd = df_sd.merge(
    student_reformulation,
    on="studId",
    how="left"
)

# rounding
df_sd[
    [
        "user_reformulation_mean",
        "user_reformulation_std"
    ]
] = (
    df_sd[
        [
            "user_reformulation_mean",
            "user_reformulation_std"
        ]
    ]
    .fillna(0)
    .round(2)
)

# ----------------------------------------------------------
# Restore original dataframe order
# ----------------------------------------------------------
df_sd = (
    df_sd
    .sort_values("_orig_order")
    .drop(columns="_orig_order")
    .reset_index(drop=True)
)

# ----------------------------------------------------------
# Summary statistics
# ----------------------------------------------------------
print("\nReformulation feature completed.")

print(
    f"\nMean normalized reformulation distance: "
    f"{df_sd['reformulation_levenshtein_norm'].mean():.3f}"
)


Computing reformulation distances...


100%|█████████████████████████████████| 11130/11130 [00:00<00:00, 165116.65it/s]


Reformulation feature completed.

Mean normalized reformulation distance: 0.682


In [119]:
df_sd.head(50)

,id,prompt,response,timestamp,studId,chatId,is_question,user_question_rate,closure,closure_words,user_closure_rate,comparative,comparative_words,user_comparative_rate,reformulation_levenshtein,reformulation_levenshtein_norm,user_reformulation_mean,user_reformulation_std
0,20377,what should i eat today?,"As a teacher of Korean history and culture, I ...",2024-03-05 09:59:00,399,135,question,0.7000,False,[],0.600000,not_comparative,[],0.500000,0,0.00,0.51,0.30
1,20384,recipe for kimchi,I can certainly guide you towards making your ...,2024-03-05 10:00:00,399,135,not_question,0.7000,False,[],0.600000,not_comparative,[],0.500000,22,0.92,0.51,0.30
2,20456,basic greetings in korea,"Absolutely! In Korean culture, greetings are v...",2024-03-05 10:05:00,399,135,not_question,0.7000,False,[],0.600000,not_comparative,[],0.500000,20,0.83,0.51,0.30
3,20528,basic words in korea,"Sure, here are some basic Korean words you can...",2024-03-05 10:06:00,399,135,not_question,0.7000,False,[],0.600000,not_comparative,[],0.500000,8,0.33,0.51,0.30
4,20546,how to say you are so pretty in korea,"In Korean, if you want to tell someone that th...",2024-03-05 10:07:00,399,135,question,0.7000,True,[are],0.600000,not_comparative,[],0.500000,23,0.62,0.51,0.30
5,20680,what are the different needs and interest duri...,"During COVID-19, people's needs and interests ...",2024-03-05 10:34:00,399,135,question,0.7000,True,"[what are, different]",0.600000,comparative,"[different, and, during]",0.500000,42,0.72,0.51,0.30
6,20732,what are the unequal sharing of needs and inte...,Certainly. COVID-19 has brought many inequalit...,2024-03-05 10:40:00,399,135,question,0.7000,True,[what are],0.600000,comparative,"[unequal, and, during]",0.500000,17,0.26,0.51,0.30
7,24190,why is the differing needs and interests a gre...,The differing needs and interests in a diverse...,2024-11-05 21:30:00,399,135,question,0.7000,True,[why],0.600000,comparative,"[differing, and, challenge]",0.500000,59,0.68,0.51,0.30
8,24191,why is the unequal sharing of costs a great ch...,Unequal sharing of costs can lead to social in...,2024-11-05 22:10:00,399,135,question,0.7000,True,"[why, what is]",0.600000,comparative,"[unequal, costs, challenge]",0.500000,50,0.54,0.51,0.30
9,24192,why is the unequal sharing of costs a great ch...,"The Singapore government, like any other, must...",2024-11-05 22:11:00,399,135,question,0.7000,True,"[why, what is]",0.600000,comparative,"[unequal, costs, challenge]",0.500000,29,0.24,0.51,0.30


### 4.5 Feature 13: Time Interval Between Prompts

This feature measures the **time gap between consecutive prompts within the same chat session**, providing a proxy for user reflection time and pacing of interaction.

For each user session (defined by `studId`, `chatId`, and `session_id`), prompts are first ordered chronologically by timestamp. The difference between each prompt and its immediate predecessor is computed to obtain the inter-prompt time interval.

#### (i) Sequential Time Difference Computation

For each session:
- Sort prompts by timestamp in ascending order
- Compute the time difference between consecutive prompts

  - First prompt in a session has no previous message → interval = 0  
  - Subsequent prompts:
  
$$
\text{time_interval}_i = t_i - t_{i-1}
$$


- The difference is converted into **minutes**

#### (ii) Average Session-Level Interval

To summarise pacing behaviour within a session:

- Compute the mean of non-zero intervals:

  avg_time_interval = mean(time_interval > 0)

- Rounded to **2 decimal places**
- First prompt is excluded from the average to avoid bias

#### Output Columns

| Column | Meaning |
|--------|--------|
| time_interval | Time difference (in minutes) from previous prompt |
| avg_time_interval | Average inter-prompt interval per session |

#### Notes

- If timestamps are missing or invalid, values are set to NaN
- Sessions with only one valid interval will have avg_time_interval = NaN
- This feature captures user reflection time:
  - Higher values → slower, more deliberate interaction
  - Lower values → rapid-fire prompting behaviour

In [121]:
import pandas as pd
import numpy as np

# =========================
# 1. Timestamp
# =========================
df_sd['timestamp'] = pd.to_datetime(df_sd['timestamp'], errors='coerce')

df_sd['_original_order'] = np.arange(len(df_sd))

df_sd = df_sd.sort_values(['studId', 'chatId', 'timestamp'])

# =========================
# 2. Session split (robust version)
#    - avoid over-splitting tiny noise
# =========================
time_diff = (
    df_sd.groupby(['studId', 'chatId'])['timestamp']
    .diff()
)

# safer thresholding: only split if gap is BOTH valid AND large
df_sd['new_session'] = time_diff.gt(pd.Timedelta(minutes=30))
df_sd['new_session'] = df_sd['new_session'].fillna(False)

df_sd['session_id'] = (
    df_sd.groupby(['studId', 'chatId'])['new_session']
    .cumsum()
)

# =========================
# 3. Time interval within session
# =========================
df_sd['time_interval'] = (
    df_sd.groupby(['studId', 'chatId', 'session_id'])['timestamp']
    .diff()
    .dt.total_seconds() / 60
)

df_sd['time_interval'] = df_sd['time_interval'].fillna(0).round(2)

# =========================
# 4. Avg interval (exclude zeros properly + avoid empty groups)
# =========================
df_sd['avg_time_interval'] = (
    df_sd.groupby(['studId', 'chatId', 'session_id'])['time_interval']
    .transform(lambda x: x[x > 0].mean() if (x > 0).any() else np.nan)
    .round(2)
)

# =========================
# 5. Restore order
# =========================
df_sd = (
    df_sd.sort_values('_original_order')
    .drop(columns=['_original_order', 'new_session', 'session_id'])
    .reset_index(drop=True)
)

df_sd.head(50)

,id,prompt,response,timestamp,studId,chatId,is_question,user_question_rate,closure,closure_words,user_closure_rate,comparative,comparative_words,user_comparative_rate,reformulation_levenshtein,reformulation_levenshtein_norm,user_reformulation_mean,user_reformulation_std,time_interval,avg_time_interval
0,20377,what should i eat today?,"As a teacher of Korean history and culture, I ...",2024-03-05 09:59:00,399,135,question,0.7000,False,[],0.600000,not_comparative,[],0.500000,0,0.00,0.51,0.30,0.0,6.83
1,20384,recipe for kimchi,I can certainly guide you towards making your ...,2024-03-05 10:00:00,399,135,not_question,0.7000,False,[],0.600000,not_comparative,[],0.500000,22,0.92,0.51,0.30,1.0,6.83
2,20456,basic greetings in korea,"Absolutely! In Korean culture, greetings are v...",2024-03-05 10:05:00,399,135,not_question,0.7000,False,[],0.600000,not_comparative,[],0.500000,20,0.83,0.51,0.30,5.0,6.83
3,20528,basic words in korea,"Sure, here are some basic Korean words you can...",2024-03-05 10:06:00,399,135,not_question,0.7000,False,[],0.600000,not_comparative,[],0.500000,8,0.33,0.51,0.30,1.0,6.83
4,20546,how to say you are so pretty in korea,"In Korean, if you want to tell someone that th...",2024-03-05 10:07:00,399,135,question,0.7000,True,[are],0.600000,not_comparative,[],0.500000,23,0.62,0.51,0.30,1.0,6.83
5,20680,what are the different needs and interest duri...,"During COVID-19, people's needs and interests ...",2024-03-05 10:34:00,399,135,question,0.7000,True,"[what are, different]",0.600000,comparative,"[different, and, during]",0.500000,42,0.72,0.51,0.30,27.0,6.83
6,20732,what are the unequal sharing of needs and inte...,Certainly. COVID-19 has brought many inequalit...,2024-03-05 10:40:00,399,135,question,0.7000,True,[what are],0.600000,comparative,"[unequal, and, during]",0.500000,17,0.26,0.51,0.30,6.0,6.83
7,24190,why is the differing needs and interests a gre...,The differing needs and interests in a diverse...,2024-11-05 21:30:00,399,135,question,0.7000,True,[why],0.600000,comparative,"[differing, and, challenge]",0.500000,59,0.68,0.51,0.30,0.0,NaN
8,24191,why is the unequal sharing of costs a great ch...,Unequal sharing of costs can lead to social in...,2024-11-05 22:10:00,399,135,question,0.7000,True,"[why, what is]",0.600000,comparative,"[unequal, costs, challenge]",0.500000,50,0.54,0.51,0.30,0.0,1.00
9,24192,why is the unequal sharing of costs a great ch...,"The Singapore government, like any other, must...",2024-11-05 22:11:00,399,135,question,0.7000,True,"[why, what is]",0.600000,comparative,"[unequal, costs, challenge]",0.500000,29,0.24,0.51,0.30,1.0,1.00


### 4.6 Feature 14: Total Prompts Per Student

This feature measures the **total number of prompts submitted by each student**, providing a high-level indicator of overall engagement and activity.

Since no session-based grouping is used for this feature, aggregation is performed at the **student level (`studId`)**.

This is equivalent to `num_query_episode` in the original pipeline, adapted to reflect student-level behaviour instead of query episodes.

#### Definition

For each student:

$$
\text{total\_prompts\_student} = \sum_{i=1}^{N} 1
$$

where \(N\) is the number of prompts submitted by the student.

#### Interpretation

- Higher values indicate **greater engagement / activity volume**
- Lower values indicate **less interaction with the system**

In [123]:
# total prompts per student (studId)
student_tally = (
    df_sd.groupby('studId')['prompt']
    .size()
    .reset_index()
    .rename(columns={'prompt': 'total_prompts_student'})
)

df_sd = df_sd.merge(student_tally, on='studId', how='left')

# total prompts per chat session (studId and chatId)
chat_tally = (
    df_sd.groupby(['studId', 'chatId'])['prompt']
    .size()
    .reset_index()
    .rename(columns={'prompt': 'total_prompts_chat'})
)

df_sd = df_sd.merge(chat_tally, on=['studId', 'chatId'], how='left')

df_sd.head(20)

,id,prompt,response,timestamp,studId,chatId,is_question,user_question_rate,closure,closure_words,...,comparative_words,user_comparative_rate,reformulation_levenshtein,reformulation_levenshtein_norm,user_reformulation_mean,user_reformulation_std,time_interval,avg_time_interval,total_prompts_student,total_prompts_chat
0,20377,what should i eat today?,"As a teacher of Korean history and culture, I ...",2024-03-05 09:59:00,399,135,question,0.7000,False,[],...,[],0.500000,0,0.00,0.51,0.30,0.0,6.83,10,10
1,20384,recipe for kimchi,I can certainly guide you towards making your ...,2024-03-05 10:00:00,399,135,not_question,0.7000,False,[],...,[],0.500000,22,0.92,0.51,0.30,1.0,6.83,10,10
2,20456,basic greetings in korea,"Absolutely! In Korean culture, greetings are v...",2024-03-05 10:05:00,399,135,not_question,0.7000,False,[],...,[],0.500000,20,0.83,0.51,0.30,5.0,6.83,10,10
3,20528,basic words in korea,"Sure, here are some basic Korean words you can...",2024-03-05 10:06:00,399,135,not_question,0.7000,False,[],...,[],0.500000,8,0.33,0.51,0.30,1.0,6.83,10,10
4,20546,how to say you are so pretty in korea,"In Korean, if you want to tell someone that th...",2024-03-05 10:07:00,399,135,question,0.7000,True,[are],...,[],0.500000,23,0.62,0.51,0.30,1.0,6.83,10,10
5,20680,what are the different needs and interest duri...,"During COVID-19, people's needs and interests ...",2024-03-05 10:34:00,399,135,question,0.7000,True,"[what are, different]",...,"[different, and, during]",0.500000,42,0.72,0.51,0.30,27.0,6.83,10,10
6,20732,what are the unequal sharing of needs and inte...,Certainly. COVID-19 has brought many inequalit...,2024-03-05 10:40:00,399,135,question,0.7000,True,[what are],...,"[unequal, and, during]",0.500000,17,0.26,0.51,0.30,6.0,6.83,10,10
7,24190,why is the differing needs and interests a gre...,The differing needs and interests in a diverse...,2024-11-05 21:30:00,399,135,question,0.7000,True,[why],...,"[differing, and, challenge]",0.500000,59,0.68,0.51,0.30,0.0,NaN,10,10
8,24191,why is the unequal sharing of costs a great ch...,Unequal sharing of costs can lead to social in...,2024-11-05 22:10:00,399,135,question,0.7000,True,"[why, what is]",...,"[unequal, costs, challenge]",0.500000,50,0.54,0.51,0.30,0.0,1.00,10,10
9,24192,why is the unequal sharing of costs a great ch...,"The Singapore government, like any other, must...",2024-11-05 22:11:00,399,135,question,0.7000,True,"[why, what is]",...,"[unequal, costs, challenge]",0.500000,29,0.24,0.51,0.30,1.0,1.00,10,10


### 4.7 Features 15–18: Question Typology (O_NP, O_P, C_NP, C_P)

This feature classifies each prompt into one of four question types based on the **first meaningful tokens** in the sentence.

The system improves robustness by handling noisy inputs, filler words, and multi-word openings such as *“how do”, “can you”, “how about”*.

Unlike a strict filtering approach, this version is designed for a **full-dataset pipeline**, where:
- All rows are retained
- Only valid samples are used for training
- Invalid cases are assigned `NaN` in ML predictions

---

#### Classification Rules

| Code | Type | Rule |
|------|------|------|
| O_NP | Open, Non-Possibility | Starts with `who / when / what / where / how / why` AND second meaningful token NOT in `{can, may, might, could, would, if, will}` |
| O_P  | Open, Possibility | Starts with `who / when / what / where / how / why` AND second meaningful token IN `{can, may, might, could, would, if, will}` |
| C_NP | Closed, Non-Possibility | Starts with `must / should / do / does / did / is / are / have / has` |
| C_P  | Closed, Possibility | Starts with `can / may / might / could / would / will` |

---

#### Improvements Over Basic Rule System

The improved version addresses limitations in naive keyword matching:

- Handles **noisy tokens** (e.g., "u", "please", "just")
- Normalizes contractions (e.g., "what's" → "what is")
- Improves second-token detection by skipping filler words
- Handles edge cases like `"how about"` as a structured phrase
- Reduces misclassification from conversational prompts
- Supports **full dataset prediction without row removal**

---

#### Pipeline Design Note (IMPORTANT)

This implementation follows a **two-stage ML workflow**:

1. **Rule-based labeling is used ONLY for training data**
   - Rows with label `"OTHER"` are excluded from training only
   - No rows are removed from the original dataset

2. **Model inference is applied to ALL rows**
   - Full dataset is preserved
   - Invalid/uncertain cases are assigned `NaN` in predictions

---

#### Token Logic

The classification is based on:

1. **First token** → determines question type group (Open / Closed / Possibility)
2. **Second meaningful token** → refines intent (especially for Open questions)

Filler words are ignored:
`you, i, we, the, a, to, me, it, this, that, please, just`

---

#### Example Interpretations

| Prompt | Label |
|--------|------|
| "How do I improve this sentence?" | O_NP |
| "How can I improve this?" | O_P |
| "Is this correct?" | C_NP |
| "Can you help me?" | C_P |
| "What if I change this?" | O_P |

---

#### Output Handling in Final Dataset

| Case | Output in `question_typology_ml` |
|------|----------------------------------|
| Valid prompt | O_NP / O_P / C_NP / C_P |
| Unclassifiable / OTHER | NaN |

---

#### Purpose

This feature is used to:

- Analyze **question intent structure**
- Distinguish between **open-ended vs directive prompts**
- Capture **modality (possibility vs non-possibility)**
- Support downstream behavioral analysis and ML classification
- Enable full-dataset export without losing information

In [126]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# ==========================================================
# 1. Rule-based labeling
# ==========================================================
OPEN_WH = {"who", "when", "what", "where", "how", "why"}
POSSIBILITY = {"can", "may", "might", "could", "would", "will", "if"}
CLOSED = {"must", "should", "do", "does", "did", "is", "are", "have", "has"}

def rule_label(text):
    if pd.isna(text):
        return "OTHER"

    text = text.lower().strip()
    tokens = re.findall(r"[a-z']+", text)

    if len(tokens) == 0:
        return "OTHER"

    first = tokens[0]
    second = tokens[1] if len(tokens) > 1 else ""

    if first in OPEN_WH:
        return "O_P" if second in POSSIBILITY else "O_NP"

    if first in CLOSED:
        return "C_NP"

    if first in POSSIBILITY:
        return "C_P"

    return "OTHER"


# ==========================================================
# 2. Create FULL labels (DO NOT DROP ANY ROWS)
# ==========================================================
df_sd["label"] = df_sd["prompt"].apply(rule_label)

# ==========================================================
# 3. Split ONLY valid training rows
# ==========================================================
train_df = df_sd[df_sd["label"] != "OTHER"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    train_df["prompt"],
    train_df["label"],
    test_size=0.2,
    random_state=42,
    stratify=train_df["label"]
)

# ==========================================================
# 4. Model
# ==========================================================
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1,2),
        lowercase=True
    )),
    ("clf", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

# ==========================================================
# 5. Train ONLY on valid labels
# ==========================================================
model.fit(X_train, y_train)

# ==========================================================
# 6. Evaluate (optional)
# ==========================================================
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# ==========================================================
# 7. Predict on FULL dataset (IMPORTANT STEP)
# ==========================================================
df_sd["question_typology_ml"] = model.predict(df_sd["prompt"])

# ==========================================================
# 8. Handle OTHER rows properly
# ==========================================================
df_sd.loc[df_sd["label"] == "OTHER", "question_typology_ml"] = np.nan

# ==========================================================
# 9. Save full dataset (NO ROW LOSS)
# ==========================================================
df_sd.to_csv("prompt_surface_deep.csv", index=False)

print("Saved full dataset with NO rows removed")

Accuracy: 0.945679012345679
              precision    recall  f1-score   support

        C_NP       0.89      0.84      0.87        88
         C_P       0.92      0.96      0.94       143
        O_NP       0.96      0.97      0.96       516
         O_P       0.93      0.90      0.92        63

    accuracy                           0.95       810
   macro avg       0.93      0.92      0.92       810
weighted avg       0.95      0.95      0.95       810

Saved full dataset with NO rows removed


In [127]:
df_sd.shape

(11130, 24)